In [6]:
import pandas as pd
import json
import re
import os

with open("../data/raw/videos_raw.json", "r", encoding="utf-8") as f:
    videos = json.load(f)

def convertir_duree(iso):
    if not iso:
        return 0
    match = re.match(r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?', iso)
    if not match:
        return 0
    heures   = int(match.group(1) or 0)
    minutes  = int(match.group(2) or 0)
    secondes = int(match.group(3) or 0)
    return heures * 3600 + minutes * 60 + secondes

def classifier_video(duree_sec):
    if duree_sec < 60:
        return "short"
    elif duree_sec < 3600:
        return "video"
    elif duree_sec < 7200:
        return "stream"
    else:
        return "podcast"

dataset = []
for video in videos:
    duree = convertir_duree(video["duration"])
    ligne = {
        "videoId"      : video["videoId"],
        "title"        : (video["title"] or "").strip(),
        "date"         : (video["publishedAt"] or "")[:10],
        "duree_sec"    : duree,
        "vues"         : int(video["viewCount"]    or 0),
        "likes"        : int(video["likeCount"]    or 0),
        "commentaires" : int(video["commentCount"] or 0),
        "type"         : classifier_video(duree),  # ✅ une colonne au lieu de 4 fichiers
    }
    dataset.append(ligne)

# ✅ Un seul fichier CSV
os.makedirs("../data/processed", exist_ok=True)
df = pd.DataFrame(dataset)
df.to_csv("../data/processed/data.csv", index=False, encoding="utf-8")

print(f"✅ Fichier sauvegardé : ../data/processed/data.csv")
print(f"   {len(df)} lignes | {len(df.columns)} colonnes : {list(df.columns)}")
print(df["type"].value_counts())

✅ Fichier sauvegardé : ../data/processed/data.csv
   968 lignes | 8 colonnes : ['videoId', 'title', 'date', 'duree_sec', 'vues', 'likes', 'commentaires', 'type']
type
video      768
short      189
podcast      9
stream       2
Name: count, dtype: int64
